# LungSeg Kaggle Phase 4/6 Standalone

Notebook/script autocontenido para ejecutar el pipeline actual en Kaggle sin
depender de `src/lungseg`, `configs/` ni de una instalacion editable del proyecto.
Replica la configuracion viva: modelo ViT 2.5D, DiceFocalLoss, entrenamiento por
epocas, inferencia por ventana deslizante, Phase 4 por folds, Phase 6 de ablacion
y Phase 5 LIDC opcional si existe un manifiesto.

Equivalente a: `python -m lungseg.cli train training=kaggle_p100 experiment=phase4_full`

## Bootstrap de dependencias

In [1]:
from __future__ import annotations

import importlib.metadata
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

CORE_REQUIREMENTS = [
    ("monai", "monai>=1.5.2"),
    ("nibabel", "nibabel>=5.0.0"),
    ("scipy", "scipy>=1.11.0"),
    ("sklearn", "scikit-learn>=1.3.0"),
    ("pandas", "pandas>=2.0.0"),
    ("tqdm", "tqdm>=4.65"),
    ("matplotlib", "matplotlib>=3.7.0"),
    ("timm", "timm>=1.0.26"),
    ("wandb", "wandb>=0.17"),
]

P100_TORCH_REQUIREMENTS = [
    "torch==2.6.0",
    "torchvision==0.21.0",
    "torchaudio==2.6.0",
    "--index-url",
    "https://download.pytorch.org/whl/cu124",
]
TORCH_RUNTIME_DEP_PREFIXES = ("nvidia-", "triton")


def _truthy(value: str | None, default: bool = True) -> bool:
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "y", "on"}


def _installed_version(package: str) -> str | None:
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return None


def _nvidia_smi_lines() -> list[str]:
    try:
        output = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
            text=True,
            stderr=subprocess.DEVNULL,
        )
    except Exception:
        return []
    return [line.strip() for line in output.splitlines() if line.strip()]


def _has_p100_gpu() -> bool:
    return any(
        "P100" in line or line.endswith(", 6.0") or ", 6.0" in line for line in _nvidia_smi_lines()
    )


def _pip_install(args: list[str]) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


def _torch_runtime_requirements() -> list[str]:
    try:
        requirements = importlib.metadata.requires("torch") or []
    except importlib.metadata.PackageNotFoundError:
        return []
    return [
        requirement
        for requirement in requirements
        if requirement.lower().startswith(TORCH_RUNTIME_DEP_PREFIXES)
    ]


def _ensure_torch_runtime_deps() -> None:
    if not _truthy(os.environ.get("INSTALL_TORCH_RUNTIME_DEPS"), default=True):
        return
    requirements = _torch_runtime_requirements()
    if requirements:
        print("Instalando dependencias runtime CUDA de PyTorch...")
        _pip_install(["--no-deps", *requirements])


def _assert_torch_importable() -> None:
    try:
        import torch as torch_module

        print(f"PyTorch import OK: torch={torch_module.__version__}")
    except ImportError as exc:
        if "libcusparseLt.so.0" not in str(exc):
            raise
        print("Falta libcusparseLt.so.0; instalando nvidia-cusparselt-cu12...")
        _pip_install(["--no-deps", "nvidia-cusparselt-cu12"])
        import torch as torch_module

        print(f"PyTorch import OK: torch={torch_module.__version__}")


def _conservative_pip_default() -> bool:
    return Path("/kaggle/working").exists() or _has_p100_gpu()


def _conservative_pip() -> bool:
    return _truthy(os.environ.get("CONSERVATIVE_PIP"), default=_conservative_pip_default())


def _ensure_core_requirements() -> None:
    if not _truthy(os.environ.get("INSTALL_DEPS"), default=True):
        print("Bootstrap de dependencias desactivado con INSTALL_DEPS=0.")
        return

    missing = [
        (module_name, requirement)
        for module_name, requirement in CORE_REQUIREMENTS
        if importlib.util.find_spec(module_name) is None
    ]
    if not missing:
        return

    print("Instalando dependencias faltantes:", ", ".join(req for _, req in missing))
    conservative = _conservative_pip()
    for module_name, requirement in missing:
        args = [requirement]
        if conservative and module_name in {"monai", "timm", "wandb"}:
            args.insert(0, "--no-deps")
        _pip_install(args)


# Kaggle puede traer un PyTorch reciente sin soporte para Pascal/P100 (sm_60).
# Si detectamos P100, fijamos una build compatible antes de importar torch.
torch_loaded = "torch" in sys.modules
if _has_p100_gpu() and _truthy(os.environ.get("FIX_P100_TORCH"), default=True):
    current_torch = _installed_version("torch")
    if current_torch is None or not current_torch.startswith("2.6.0") or "+cpu" in current_torch:
        print(
            "GPU P100 detectada; instalando PyTorch compatible con sm_60 "
            f"(torch actual: {current_torch})."
        )
        _pip_install(["--force-reinstall", "--no-deps", *P100_TORCH_REQUIREMENTS])
        if torch_loaded:
            raise RuntimeError(
                "PyTorch ya estaba importado. Reinicia el kernel y ejecuta el notebook "
                "desde el principio."
            )
    _ensure_torch_runtime_deps()
    if not torch_loaded:
        _assert_torch_importable()
elif importlib.util.find_spec("torch") is None and _truthy(
    os.environ.get("INSTALL_DEPS"), default=True
):
    _pip_install(["torch>=2.0.0"])

_ensure_core_requirements()
print("Dependencias principales listas.")

Instalando dependencias runtime CUDA de PyTorch...
PyTorch import OK: torch=2.6.0+cu124
Dependencias principales listas.


## Imports y parametros globales

In [2]:
import copy
import csv
import gc
import json
import logging
import math
import random
import shutil
import tarfile
import urllib.request
from collections.abc import Iterable, Iterator, Sequence
from contextlib import nullcontext
from datetime import datetime
from functools import partial
from itertools import pairwise
from typing import Any

import nibabel as nib
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as functional
from monai.data import CacheDataset, DataLoader, Dataset, PersistentDataset
from monai.inferers import sliding_window_inference
from monai.losses import DiceFocalLoss
from monai.transforms import (
    Compose,
    CropForegroundd,
    EnsureChannelFirstd,
    EnsureTyped,
    LoadImaged,
    Orientationd,
    RandCropByPosNegLabeld,
    RandFlipd,
    RandGaussianNoised,
    RandGaussianSmoothd,
    RandRotate90d,
    RandScaleIntensityd,
    RandShiftIntensityd,
    ScaleIntensityRanged,
    Spacingd,
)
from monai.utils.misc import set_determinism
from scipy import ndimage
from scipy.stats import wilcoxon
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from torch.amp import GradScaler
from torch.optim.lr_scheduler import LambdaLR
from tqdm.auto import tqdm


def get_logger(name: str = "standalone_lungseg") -> logging.Logger:
    logger = logging.getLogger(name)
    if logger.handlers:
        return logger
    logger.setLevel(logging.INFO)
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(
        logging.Formatter("[%(asctime)s] %(levelname)s %(name)s :: %(message)s", "%H:%M:%S")
    )
    logger.addHandler(handler)
    logger.propagate = False
    return logger


def wandb_enabled() -> bool:
    return bool(os.environ.get("WANDB_API_KEY"))


LOGGER = get_logger()
IS_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path(os.environ.get("WORK_DIR", "/kaggle/working" if IS_KAGGLE else ".")).resolve()
RUN_ID = os.environ.get("RUN_ID", datetime.now().strftime("%Y%m%d-%H%M%S"))
OUTPUTS_ROOT = Path(
    os.environ.get("OUTPUTS_ROOT", str(WORK_DIR / "outputs" / "full-pipeline" / RUN_ID))
).resolve()
SPLITS_DIR = Path(os.environ.get("SPLITS_DIR", str(WORK_DIR / "data" / "splits"))).resolve()
CACHE_DIR = Path(
    os.environ.get("MONAI_CACHE_DIR", str(WORK_DIR / "data" / "cache" / "monai"))
).resolve()


def cuda_device_usable() -> bool:
    if not torch.cuda.is_available():
        return False
    try:
        capability = torch.cuda.get_device_capability(0)
        archs = set(torch.cuda.get_arch_list())
        if capability == (6, 0) and "sm_60" not in archs:
            LOGGER.warning(
                "La GPU es P100/sm_60, pero este PyTorch no incluye sm_60 (%s). "
                "Ejecuta la celda de bootstrap y reinicia el kernel si reinstalo torch.",
                sorted(archs),
            )
            return False
        torch.empty(1, device="cuda")
        return True
    except Exception as exc:
        LOGGER.warning("CUDA no esta usable en este runtime; se usara CPU. Detalle: %s", exc)
        return False


CUDA_AVAILABLE = cuda_device_usable()
if CUDA_AVAILABLE:
    print(
        "CUDA usable: "
        f"{torch.cuda.get_device_name(0)} | torch={torch.__version__} | cuda={torch.version.cuda}"
    )
else:
    print(f"CUDA no usable. torch={torch.__version__}; revisa el bootstrap si esperabas usar GPU.")


def env_int(name: str, default: int) -> int:
    return int(os.environ.get(name, default))


def env_float(name: str, default: float) -> float:
    return float(os.environ.get(name, default))


def env_bool(name: str, default: bool) -> bool:
    return _truthy(os.environ.get(name), default=default)


def env_optional_int(name: str) -> int | None:
    value = os.environ.get(name, "").strip()
    return int(value) if value else None


def env_optional_str(name: str) -> str | None:
    value = os.environ.get(name, "").strip()
    return value or None


def env_list(name: str, default: Sequence[Any], cast=str) -> list[Any]:
    value = os.environ.get(name, "").strip()
    if not value:
        return list(default)
    return [cast(item) for item in value.replace(",", " ").split()]


SEED = env_int("SEED", 42)
N_FOLDS = env_int("N_FOLDS", 5)
FOLDS = env_list("FOLDS", list(range(N_FOLDS)), int)
PHASE4_FOLDS = env_list("PHASE4_FOLDS", FOLDS, int)
PHASE6_FOLDS = env_list("PHASE6_FOLDS", FOLDS, int)

TRAINING_PROFILE = os.environ.get("TRAINING", "kaggle_p100")
MODEL_NAME = os.environ.get("MODEL", "vit_25d_lung")
MODEL_ENCODER = os.environ.get("MODEL_ENCODER", "vit_base_patch16_224")
MODEL_PRETRAINED = env_bool("MODEL_PRETRAINED", True)
ALLOW_RANDOM_INIT_FALLBACK = env_bool("ALLOW_RANDOM_INIT_FALLBACK", False)

UNFREEZE_EPOCH = env_int("UNFREEZE_EPOCH", -1)
UNFREEZE_LR_FACTOR = env_float("UNFREEZE_LR_FACTOR", 0.1)

PHASE4_MAX_ITER = env_optional_int("PHASE4_MAX_ITER")
PHASE4_VAL_EVERY = env_optional_int("PHASE4_VAL_EVERY")
PHASE4_VAL_EVERY_EPOCHS = env_optional_int("PHASE4_VAL_EVERY_EPOCHS")
PHASE4_PATIENCE = env_optional_int("PHASE4_PATIENCE")
PHASE4_RESUME_FROM = env_optional_str("PHASE4_RESUME_FROM")
PHASE4_SW_BATCH_SIZE = env_optional_int("PHASE4_SW_BATCH_SIZE")

DEFAULT_CACHE_MODE = "none" if IS_KAGGLE and not env_bool("ALLOW_DISK_CACHE", False) else "disk"
PHASE4_CACHE_MODE = os.environ.get("PHASE4_CACHE_MODE", DEFAULT_CACHE_MODE)
PHASE4_CACHE_RATE = env_float("PHASE4_CACHE_RATE", 0.0 if PHASE4_CACHE_MODE == "none" else 1.0)
PHASE4_CACHE_WORKERS = env_int("PHASE4_CACHE_WORKERS", 2)
PHASE4_NUM_WORKERS = env_int("PHASE4_NUM_WORKERS", 2)
PHASE4_PIN_MEMORY = env_bool("PHASE4_PIN_MEMORY", CUDA_AVAILABLE)

PHASE6_MAX_ITER = env_optional_int("PHASE6_MAX_ITER")
PHASE6_VAL_EVERY = env_optional_int("PHASE6_VAL_EVERY")
PHASE6_VAL_EVERY_EPOCHS = env_optional_int("PHASE6_VAL_EVERY_EPOCHS")
PHASE6_PATIENCE = env_optional_int("PHASE6_PATIENCE")
PHASE6_SW_BATCH_SIZE = env_optional_int("PHASE6_SW_BATCH_SIZE")
PHASE6_CACHE_MODE = os.environ.get("PHASE6_CACHE_MODE", PHASE4_CACHE_MODE)
PHASE6_CACHE_RATE = env_float("PHASE6_CACHE_RATE", 0.0 if PHASE6_CACHE_MODE == "none" else 1.0)
PHASE6_CACHE_WORKERS = env_int("PHASE6_CACHE_WORKERS", PHASE4_CACHE_WORKERS)
PHASE6_NUM_WORKERS = env_int("PHASE6_NUM_WORKERS", PHASE4_NUM_WORKERS)
PHASE6_PIN_MEMORY = env_bool("PHASE6_PIN_MEMORY", PHASE4_PIN_MEMORY)

RESET_MONAI_CACHE = env_bool("RESET_MONAI_CACHE", False)
FRACTIONS = env_list("FRACTIONS", [0.25, 0.5, 1.0], float)
AUGS = env_list("AUGS", ["none", "standard"], str)
SEEDS = env_list("SEEDS", [0, 1, 2], int)

RUN_PHASE4 = env_bool("RUN_PHASE4", True)
RUN_PHASE6 = env_bool("RUN_PHASE6", True)
RUN_PHASE5 = env_bool("RUN_PHASE5", True)
PHASE5_E2E = env_bool("PHASE5_E2E", False)
LIDC_MANIFEST = Path(
    os.environ.get(
        "LIDC_MANIFEST",
        str(WORK_DIR / "data" / "processed" / "lidc" / "nodule_manifest.csv"),
    )
).resolve()

print(f"WORK_DIR={WORK_DIR}")
print(f"OUTPUTS_ROOT={OUTPUTS_ROOT}")
print(f"TRAINING={TRAINING_PROFILE} MODEL={MODEL_NAME}")

<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
2026-04-30 17:52:29.420842: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777571549.597637     177 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777571549.656154     177 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777571550.087859     177 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777571550.087899     177 computation_placer.cc:1

CUDA usable: Tesla P100-PCIE-16GB | torch=2.6.0+cu124 | cuda=12.4
WORK_DIR=/kaggle/working
OUTPUTS_ROOT=/kaggle/working/outputs/full-pipeline/20260430-175243
TRAINING=kaggle_p100 MODEL=vit_25d_lung


## Configuracion embebida equivalente a Hydra

In [3]:
class Cfg(dict):
    def __init__(self, *args, **kwargs) -> None:
        super().__init__()
        self.update(*args, **kwargs)

    def __getattr__(self, key: str) -> Any:
        try:
            return self[key]
        except KeyError as exc:
            raise AttributeError(key) from exc

    def __setattr__(self, key: str, value: Any) -> None:
        self[key] = value

    def __setitem__(self, key: str, value: Any) -> None:
        super().__setitem__(key, cfgify(value))

    def update(self, *args, **kwargs) -> None:
        for key, value in dict(*args, **kwargs).items():
            self[key] = value


def cfgify(value: Any) -> Any:
    if isinstance(value, Cfg):
        return value
    if isinstance(value, dict):
        return Cfg(value)
    if isinstance(value, list):
        return [cfgify(item) for item in value]
    return value


def to_plain(value: Any) -> Any:
    if isinstance(value, Cfg):
        return {key: to_plain(item) for key, item in value.items()}
    if isinstance(value, dict):
        return {key: to_plain(item) for key, item in value.items()}
    if isinstance(value, list):
        return [to_plain(item) for item in value]
    if isinstance(value, tuple):
        return [to_plain(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    return value


def select(cfg: Cfg | dict, key: str, default: Any = None) -> Any:
    current: Any = cfg
    for part in key.split("."):
        if isinstance(current, dict) and part in current:
            current = current[part]
        else:
            return default
    return current


def update_cfg(cfg: Cfg, key: str, value: Any, force_add: bool = True) -> None:
    current = cfg
    parts = key.split(".")
    for part in parts[:-1]:
        if part not in current:
            if not force_add:
                raise KeyError(key)
            current[part] = Cfg()
        current = current[part]
    current[parts[-1]] = value


DATA_TASK06 = {
    "name": "task06",
    "root": str(WORK_DIR / "data" / "raw" / "Task06_Lung"),
    "dataset_json": str(WORK_DIR / "data" / "raw" / "Task06_Lung" / "dataset.json"),
    "n_folds": 5,
    "splits_dir": str(SPLITS_DIR),
    "target_spacing": [0.79, 0.79, 1.24],
    "hu_clip": {"a_min": -1024, "a_max": 400, "b_min": 0.0, "b_max": 1.0, "clip": True},
    "crop_foreground": {"source_key": "image", "threshold": 0.1},
    "sampler": {"pos": 2, "neg": 1, "num_samples": 4},
    "cache": {
        "mode": "disk",
        "rate": 1.0,
        "num_workers": 2,
        "disk_dir": str(CACHE_DIR),
    },
}

DATA_LIDC = {
    "name": "lidc",
    "root": str(WORK_DIR / "data" / "raw" / "LIDC-IDRI"),
    "manifest": str(LIDC_MANIFEST),
    "pred_masks_dir": str(WORK_DIR / "data" / "processed" / "lidc" / "pred_masks"),
    "n_folds": 5,
    "malignancy_consensus": {"malignant_min": 4, "benign_max": 2},
}

MODEL_CONFIGS = {
    "vit_25d_lung": {
        "name": "vit_25d",
        "spatial_dims": 3,
        "in_channels": 1,
        "out_channels": 2,
        "encoder_name": MODEL_ENCODER,
        "pretrained": MODEL_PRETRAINED,
        "loss": {
            "name": "dice_focal",
            "to_onehot_y": True,
            "softmax": True,
            "include_background": False,
            "squared_pred": True,
            "gamma": 2.0,
            "lambda_dice": 1.0,
            "lambda_focal": 1.0,
        },
    },
}

TRAINING_CONFIGS = {
    "kaggle_p100": {
        "name": "kaggle_p100",
        "patch_size": [96, 96, 96],
        "batch_size": 1,
        "num_workers": 2,
        "amp": True,
        "pin_memory": True,
        "resume_from": None,
        "optimizer": {"name": "adamw", "lr": 1.0e-4, "weight_decay": 1.0e-5},
        "scheduler": {"name": "poly", "exp": 0.9},
        "augment_regime": "standard",
        "inference": {
            "sw_batch_size": 1,
            "overlap": 0.5,
            "mode": "gaussian",
            "padding_mode": "constant",
        },
    },
    "local_5060": {
        "name": "local_5060",
        "patch_size": [96, 96, 96],
        "batch_size": 2,
        "num_workers": 4,
        "amp": True,
        "pin_memory": True,
        "resume_from": None,
        "unfreeze_epoch": -1,
        "unfreeze_lr_factor": 0.1,
        "val_every_epochs": 1,
        "optimizer": {"name": "adamw", "lr": 1.0e-4, "weight_decay": 1.0e-5},
        "scheduler": {"name": "poly", "exp": 0.9},
        "augment_regime": "standard",
        "inference": {
            "sw_batch_size": 4,
            "overlap": 0.5,
            "mode": "gaussian",
            "padding_mode": "constant",
        },
    },
    "sanity": {
        "name": "sanity",
        "patch_size": [96, 96, 96],
        "batch_size": 2,
        "num_workers": 0,
        "amp": False,
        "pin_memory": False,
        "resume_from": None,
        "optimizer": {"name": "adamw", "lr": 3.0e-4, "weight_decay": 0.0},
        "scheduler": {"name": "poly", "exp": 0.9},
        "augment_regime": "none",
        "inference": {
            "sw_batch_size": 1,
            "overlap": 0.25,
            "mode": "gaussian",
            "padding_mode": "constant",
        },
        "sanity": {"overfit_one_batch": True, "max_iterations": 200, "val_every": 50},
    },
}

EXPERIMENT_CONFIGS = {
    "phase4_full": {
        "name": "phase4_full",
        "max_iterations": 50000,
        "val_every": 500,
        "patience": 20,
        "grad_accum_steps": 1,
        "log_every": 50,
    },
    "phase6_ablation": {
        "name": "phase6_ablation",
        "max_iterations": 10000,
        "val_every": 200,
        "patience": 10,
        "grad_accum_steps": 1,
        "log_every": 50,
        "fixed_iterations": True,
        "sweep": {
            "data_fraction": [0.25, 0.5, 1.0],
            "aug_regime": ["none", "standard"],
            "seed": [0, 1, 2],
        },
    },
}


def make_cfg(
    *,
    experiment_name: str,
    fold: int = 0,
    outputs: str | Path | None = None,
    seed: int = SEED,
    training_profile: str = TRAINING_PROFILE,
    model_name: str = MODEL_NAME,
    data_name: str = "task06",
) -> Cfg:
    if model_name not in MODEL_CONFIGS:
        raise ValueError(f"Modelo desconocido: {model_name}. Opciones: {sorted(MODEL_CONFIGS)}")
    if training_profile not in TRAINING_CONFIGS:
        raise ValueError(
            "Perfil de entrenamiento desconocido: "
            f"{training_profile}. Opciones: {sorted(TRAINING_CONFIGS)}"
        )
    if experiment_name not in EXPERIMENT_CONFIGS:
        raise ValueError(
            f"Experimento desconocido: {experiment_name}. Opciones: {sorted(EXPERIMENT_CONFIGS)}"
        )

    data = copy.deepcopy(DATA_LIDC if data_name == "lidc" else DATA_TASK06)
    cfg = Cfg(
        {
            "seed": int(seed),
            "fold": int(fold),
            "data": data,
            "model": copy.deepcopy(MODEL_CONFIGS[model_name]),
            "training": copy.deepcopy(TRAINING_CONFIGS[training_profile]),
            "experiment": copy.deepcopy(EXPERIMENT_CONFIGS[experiment_name]),
            "paths": {
                "outputs": str(outputs or (OUTPUTS_ROOT / experiment_name)),
                "splits": str(SPLITS_DIR),
            },
        }
    )
    cfg.training.unfreeze_epoch = UNFREEZE_EPOCH
    cfg.training.unfreeze_lr_factor = UNFREEZE_LR_FACTOR

    if experiment_name == "phase4_full":
        cfg.data.cache.mode = PHASE4_CACHE_MODE
        cfg.data.cache.rate = PHASE4_CACHE_RATE
        cfg.data.cache.num_workers = PHASE4_CACHE_WORKERS
        cfg.training.num_workers = PHASE4_NUM_WORKERS
        cfg.training.pin_memory = PHASE4_PIN_MEMORY
        if PHASE4_RESUME_FROM:
            cfg.training.resume_from = PHASE4_RESUME_FROM
        if PHASE4_SW_BATCH_SIZE is not None:
            cfg.training.inference.sw_batch_size = PHASE4_SW_BATCH_SIZE
        if PHASE4_MAX_ITER is not None:
            cfg.experiment.max_iterations = PHASE4_MAX_ITER
        if PHASE4_VAL_EVERY is not None:
            cfg.experiment.val_every = PHASE4_VAL_EVERY
        if PHASE4_VAL_EVERY_EPOCHS is not None:
            cfg.training.val_every_epochs = PHASE4_VAL_EVERY_EPOCHS
        if PHASE4_PATIENCE is not None:
            cfg.experiment.patience = PHASE4_PATIENCE

    if experiment_name == "phase6_ablation":
        cfg.data.cache.mode = PHASE6_CACHE_MODE
        cfg.data.cache.rate = PHASE6_CACHE_RATE
        cfg.data.cache.num_workers = PHASE6_CACHE_WORKERS
        cfg.training.num_workers = PHASE6_NUM_WORKERS
        cfg.training.pin_memory = PHASE6_PIN_MEMORY
        if PHASE6_SW_BATCH_SIZE is not None:
            cfg.training.inference.sw_batch_size = PHASE6_SW_BATCH_SIZE
        if PHASE6_MAX_ITER is not None:
            cfg.experiment.max_iterations = PHASE6_MAX_ITER
        if PHASE6_VAL_EVERY is not None:
            cfg.experiment.val_every = PHASE6_VAL_EVERY
        if PHASE6_VAL_EVERY_EPOCHS is not None:
            cfg.training.val_every_epochs = PHASE6_VAL_EVERY_EPOCHS
        if PHASE6_PATIENCE is not None:
            cfg.experiment.patience = PHASE6_PATIENCE

    return cfg

## Task06 y splits patient-level

In [4]:
TASK06_URL = os.environ.get(
    "TASK06_URL", "https://msd-for-monai.s3-us-west-2.amazonaws.com/Task06_Lung.tar"
)
TASK06_ROOT: Path | None = None
TASK06_JSON: Path | None = None


def _safe_extract_tar(tar_path: Path, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    dest_resolved = destination.resolve()
    try:
        with tarfile.open(tar_path, mode="r:*") as handle:
            members = handle.getmembers()
            for member in members:
                if member.issym() or member.islnk():
                    raise RuntimeError(f"Entrada con enlace no permitida en tar: {member.name}")
                target = (destination / member.name).resolve()
                try:
                    target.relative_to(dest_resolved)
                except ValueError as exc:
                    raise RuntimeError(f"Entrada insegura en tar: {member.name}") from exc
            handle.extractall(destination, members=members)
    except (tarfile.TarError, EOFError, OSError) as exc:
        raise RuntimeError(
            f"No se pudo leer {tar_path}; el archivo parece incompleto o corrupto"
        ) from exc


def _download_task06_archive() -> Path | None:
    if not env_bool("TASK06_DOWNLOAD", True):
        print("Descarga automatica de Task06 desactivada con TASK06_DOWNLOAD=0.")
        return None

    extract_base = Path(os.environ.get("TASK06_DOWNLOAD_ROOT", str(WORK_DIR / "data" / "raw")))
    task_root = extract_base / "Task06_Lung"
    if (task_root / "dataset.json").exists():
        return task_root.resolve()

    archive_path = Path(os.environ.get("TASK06_TAR_DOWNLOAD", str(WORK_DIR / "Task06_Lung.tar")))
    try:
        print(f"No se encontro Task06 localmente. Descargando {TASK06_URL}...")
        archive_path.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(TASK06_URL, archive_path)
        print(f"Extrayendo {archive_path} en {extract_base}...")
        _safe_extract_tar(archive_path, extract_base)
        archive_path.unlink(missing_ok=True)
    except Exception as exc:
        print(f"No se pudo descargar/extractar Task06_Lung: {exc}")
        return None

    if (task_root / "dataset.json").exists():
        return task_root.resolve()
    return None


def locate_task06_root() -> Path:
    env_root = os.environ.get("TASK06_ROOT", "").strip()
    candidates: list[Path] = []
    if env_root:
        candidates.append(Path(env_root))
    cwd = Path.cwd().resolve()
    candidates.extend(
        [
            WORK_DIR / "data" / "raw" / "Task06_Lung",
            WORK_DIR / "Task06_Lung",
            cwd / "data" / "raw" / "Task06_Lung",
            cwd / "Task06_Lung",
            cwd / "notebooks" / "Task06_Lung",
            cwd.parent / "data" / "raw" / "Task06_Lung",
            cwd.parent / "Task06_Lung",
        ]
    )
    for candidate in candidates:
        if (candidate / "dataset.json").exists():
            return candidate.resolve()

    env_tar = os.environ.get("TASK06_TAR", "").strip()
    tar_candidates: list[Path] = []
    if env_tar:
        tar_candidates.append(Path(env_tar))
    tar_candidates.extend(
        [
            WORK_DIR / "data" / "raw" / "Task06_Lung.tar",
            WORK_DIR / "Task06_Lung.tar",
            cwd / "data" / "raw" / "Task06_Lung.tar",
            cwd / "Task06_Lung.tar",
            cwd / "notebooks" / "Task06_Lung.tar",
            cwd.parent / "data" / "raw" / "Task06_Lung.tar",
            cwd.parent / "Task06_Lung.tar",
        ]
    )
    input_dir = Path("/kaggle/input")
    if input_dir.exists():
        tar_candidates.extend(sorted(input_dir.rglob("Task06_Lung.tar")))

    seen_tar_paths: set[Path] = set()
    for tar_path in tar_candidates:
        tar_path = tar_path.expanduser().resolve()
        if tar_path in seen_tar_paths or not tar_path.exists():
            continue
        seen_tar_paths.add(tar_path)
        extract_base = WORK_DIR / "data" / "raw"
        print(f"Extrayendo {tar_path} en {extract_base}...")
        try:
            _safe_extract_tar(tar_path, extract_base)
        except RuntimeError as exc:
            print(f"Aviso: {exc}. Probando otra fuente de Task06_Lung...")
            continue
        extracted = extract_base / "Task06_Lung"
        if (extracted / "dataset.json").exists():
            return extracted.resolve()

    if input_dir.exists():
        for dataset_json in sorted(input_dir.rglob("dataset.json")):
            root = dataset_json.parent
            if (
                root.name == "Task06_Lung"
                and (root / "imagesTr").exists()
                and (root / "labelsTr").exists()
            ):
                return root.resolve()

    downloaded_root = _download_task06_archive()
    if downloaded_root is not None:
        return downloaded_root

    raise FileNotFoundError(
        "No encuentro Task06_Lung. En Kaggle anade el dataset con la carpeta Task06_Lung "
        "o define TASK06_ROOT/TASK06_TAR antes de ejecutar el notebook. Si quieres forzar "
        "la descarga, deja TASK06_DOWNLOAD=1; si Kaggle no tiene internet, anade el dataset "
        "como input."
    )


def check_task06(root: Path) -> Path:
    dataset_json = root / "dataset.json"
    if not dataset_json.exists():
        raise FileNotFoundError(f"Falta {dataset_json}")
    if not (root / "imagesTr").is_dir():
        raise FileNotFoundError(f"Falta {root / 'imagesTr'}")
    if not (root / "labelsTr").is_dir():
        raise FileNotFoundError(f"Falta {root / 'labelsTr'}")
    images = sorted((root / "imagesTr").glob("*.nii.gz"))
    labels = sorted((root / "labelsTr").glob("*.nii.gz"))
    print(f"Task06 OK: {len(images)} imagenes de train, {len(labels)} etiquetas. Root={root}")
    if not labels:
        raise RuntimeError(f"No hay etiquetas en {root / 'labelsTr'}")
    return dataset_json


def _resolve_dataset_path(path_in_json: str, dataset_dir: Path) -> Path:
    return (dataset_dir / path_in_json.lstrip("./")).resolve()


def _portable_case_path(path: Path) -> str:
    resolved = path.resolve()
    try:
        return str(resolved.relative_to(WORK_DIR))
    except ValueError:
        return str(resolved)


def _compute_tumor_volume(label_abs: Path) -> float:
    img = nib.load(str(label_abs))
    voxel_volume = float(np.prod(img.header.get_zooms()[:3]))
    data = np.asarray(img.dataobj)
    n_voxels = float((data > 0).sum())
    return n_voxels * voxel_volume


def _assign_strata(volumes: np.ndarray) -> np.ndarray:
    edges = np.percentile(volumes, [100 / 3.0, 200 / 3.0])
    return np.digitize(volumes, edges).astype(int)


def make_splits(dataset_json: Path, out_dir: Path, seed: int = 42, k: int = 5) -> list[Path]:
    dataset_json = Path(dataset_json)
    out_dir = Path(out_dir)
    dataset_dir = dataset_json.parent
    contents = json.loads(dataset_json.read_text(encoding="utf-8"))

    images: list[str] = []
    labels: list[str] = []
    patient_ids: list[str] = []
    volumes: list[float] = []

    for entry in tqdm(contents["training"], desc="Calculando volumen tumoral"):
        image_abs = _resolve_dataset_path(entry["image"], dataset_dir)
        label_abs = _resolve_dataset_path(entry["label"], dataset_dir)
        patient = Path(entry["image"]).name.replace(".nii.gz", "")
        images.append(_portable_case_path(image_abs))
        labels.append(_portable_case_path(label_abs))
        patient_ids.append(patient)
        volumes.append(_compute_tumor_volume(label_abs))

    volumes_arr = np.asarray(volumes, dtype=np.float64)
    strata = _assign_strata(volumes_arr)
    cases = [
        {
            "image": images[i],
            "label": labels[i],
            "patient_id": patient_ids[i],
            "tumor_volume_mm3": round(float(volumes_arr[i]), 3),
            "stratum": int(strata[i]),
        }
        for i in range(len(images))
    ]

    splitter = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed)
    out_dir.mkdir(parents=True, exist_ok=True)
    out_paths: list[Path] = []
    dummy_x = np.zeros(len(cases))

    for fold_idx, (train_idx, val_idx) in enumerate(
        splitter.split(dummy_x, y=strata, groups=np.asarray(patient_ids))
    ):
        train = sorted([cases[i] for i in train_idx], key=lambda c: c["patient_id"])
        val = sorted([cases[i] for i in val_idx], key=lambda c: c["patient_id"])
        payload = {
            "fold": fold_idx,
            "seed": seed,
            "k": k,
            "n_train": len(train),
            "n_val": len(val),
            "train": train,
            "val": val,
        }
        out_path = out_dir / f"fold_{fold_idx}.json"
        out_path.write_text(json.dumps(payload, indent=2, sort_keys=False) + "\n", encoding="utf-8")
        out_paths.append(out_path)

    return out_paths


def check_splits(splits_dir: Path, folds: Sequence[int]) -> None:
    missing = [fold for fold in folds if not (splits_dir / f"fold_{fold}.json").exists()]
    if missing:
        raise FileNotFoundError(f"Faltan splits para folds: {missing}")
    print(f"Splits OK: {list(folds)} en {splits_dir}")


def prepare_task06() -> None:
    global TASK06_JSON, TASK06_ROOT

    TASK06_ROOT = locate_task06_root()
    TASK06_JSON = check_task06(TASK06_ROOT)
    DATA_TASK06["root"] = str(TASK06_ROOT)
    DATA_TASK06["dataset_json"] = str(TASK06_JSON)
    DATA_TASK06["splits_dir"] = str(SPLITS_DIR)

    split_paths = make_splits(TASK06_JSON, SPLITS_DIR, seed=SEED, k=N_FOLDS)
    print("Splits generados:")
    for path in split_paths:
        print(" -", path)
    check_splits(SPLITS_DIR, FOLDS)

## Transforms, datos e inferencia

In [5]:
KEYS = ["image", "label"]
CROP_METADATA_KEYS = ["foreground_start_coord", "foreground_end_coord"]
AUG_PROB = {"none": 0.0, "standard": 0.15, "aggressive": 0.30}
CACHE_MODE_ALIASES = {
    "0": "none",
    "false": "none",
    "off": "none",
    "no": "none",
    "none": "none",
    "uncached": "none",
    "auto": "auto",
    "1": "ram",
    "true": "ram",
    "cache": "ram",
    "cached": "ram",
    "memory": "ram",
    "ram": "ram",
    "persistent": "disk",
    "persistentdataset": "disk",
    "disk": "disk",
}


def set_global_determinism(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if CUDA_AVAILABLE:
        torch.cuda.manual_seed_all(seed)
    set_determinism(seed=seed)


def seed_worker(worker_id: int) -> None:
    del worker_id
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def _check_no_lr_flip(transforms: list) -> None:
    for transform in transforms:
        if isinstance(transform, RandFlipd):
            axis = transform.flipper.spatial_axis
            if axis is None:
                raise ValueError(
                    "RandFlipd sin spatial_axis voltea todos los ejes, incluido LR. "
                    "Usa spatial_axis=2 en TC de torax."
                )
            axes = (axis,) if isinstance(axis, int) else tuple(axis)
            if 0 in axes:
                raise ValueError(
                    "RandFlipd spatial_axis=0 (LR) esta prohibido en TC de torax. "
                    "Usa spatial_axis=2."
                )


def _pre_transforms(cfg: Cfg, with_label: bool = True) -> list:
    keys = list(KEYS) if with_label else ["image"]
    spacing_modes = ("bilinear", "nearest") if with_label else ("bilinear",)
    return [
        LoadImaged(keys=keys),
        EnsureChannelFirstd(keys=keys),
        Orientationd(keys=keys, axcodes="RAS"),
        Spacingd(keys=keys, pixdim=tuple(cfg.data.target_spacing), mode=spacing_modes),
        ScaleIntensityRanged(
            keys=["image"],
            a_min=float(cfg.data.hu_clip.a_min),
            a_max=float(cfg.data.hu_clip.a_max),
            b_min=float(cfg.data.hu_clip.b_min),
            b_max=float(cfg.data.hu_clip.b_max),
            clip=bool(cfg.data.hu_clip.clip),
        ),
        CropForegroundd(
            keys=keys,
            source_key="image",
            select_fn=lambda x, t=float(cfg.data.crop_foreground.threshold): x > t,
            allow_smaller=True,
        ),
    ]


def _rotate90_augmentation(prob: float, patch_size: Sequence[int]) -> list:
    if len(patch_size) < 2 or int(patch_size[0]) == int(patch_size[1]):
        return [RandRotate90d(keys=KEYS, prob=prob, max_k=3, spatial_axes=(0, 1))]
    LOGGER.warning(
        "RandRotate90d desactivado: patch_size=%s no es cuadrado en los ejes (0, 1).",
        list(patch_size),
    )
    return []


def _augmentations(prob: float, patch_size: Sequence[int]) -> list:
    if prob <= 0.0:
        return []
    return [
        RandFlipd(keys=KEYS, prob=prob, spatial_axis=2),
        *_rotate90_augmentation(prob, patch_size),
        RandGaussianNoised(keys=["image"], prob=prob, mean=0.0, std=0.02),
        RandGaussianSmoothd(
            keys=["image"],
            prob=prob,
            sigma_x=(0.5, 1.0),
            sigma_y=(0.5, 1.0),
            sigma_z=(0.5, 1.0),
        ),
        RandScaleIntensityd(keys=["image"], factors=0.10, prob=prob),
        RandShiftIntensityd(keys=["image"], offsets=0.10, prob=prob),
    ]


def build_train_transforms(cfg: Cfg) -> Compose:
    regime = str(cfg.training.augment_regime)
    if regime not in AUG_PROB:
        raise ValueError(f"augment_regime desconocido: {regime!r}; esperado {list(AUG_PROB)}")
    crop = RandCropByPosNegLabeld(
        keys=KEYS,
        label_key="label",
        spatial_size=tuple(cfg.training.patch_size),
        pos=float(cfg.data.sampler.pos),
        neg=float(cfg.data.sampler.neg),
        num_samples=int(cfg.data.sampler.num_samples),
        image_key="image",
        image_threshold=0.0,
        allow_smaller=True,
    )
    transforms = [
        *_pre_transforms(cfg),
        crop,
        *_augmentations(AUG_PROB[regime], cfg.training.patch_size),
        EnsureTyped(keys=[*KEYS, *CROP_METADATA_KEYS], allow_missing_keys=True),
    ]
    _check_no_lr_flip(transforms)
    return Compose(transforms)


def build_val_transforms(cfg: Cfg, with_label: bool = True) -> Compose:
    keys = list(KEYS) if with_label else ["image"]
    return Compose(
        [
            *_pre_transforms(cfg, with_label=with_label),
            EnsureTyped(keys=[*keys, *CROP_METADATA_KEYS], allow_missing_keys=True),
        ]
    )


def _resolve_record_path(value: str | Path) -> str:
    path = Path(value)
    if path.is_absolute():
        return str(path)
    return str((WORK_DIR / path).resolve())


def _load_fold(splits_dir: Path, fold: int) -> tuple[list[dict], list[dict]]:
    fold_path = splits_dir / f"fold_{fold}.json"
    payload = json.loads(fold_path.read_text(encoding="utf-8"))

    def absolutize(records: list[dict]) -> list[dict]:
        return [
            {
                **record,
                "image": _resolve_record_path(record["image"]),
                "label": _resolve_record_path(record["label"]),
            }
            for record in records
        ]

    return absolutize(payload["train"]), absolutize(payload["val"])


def _cache_mode(cfg: Cfg, cache_rate: float) -> str:
    raw_mode = select(cfg, "data.cache.mode", "auto")
    normalized = str(raw_mode).lower().replace("_", "").replace("-", "")
    mode = CACHE_MODE_ALIASES.get(normalized)
    if mode is None:
        raise ValueError(
            f"data.cache.mode desconocido={raw_mode!r}; esperado uno de auto, none, ram, disk"
        )
    if mode == "auto":
        return "ram" if cache_rate > 0.0 else "none"
    return mode


def _resolve_cache_dir(cfg: Cfg, fold: int) -> Path:
    raw_dir = select(cfg, "data.cache.disk_dir", str(CACHE_DIR))
    cache_dir = Path(str(raw_dir))
    if not cache_dir.is_absolute():
        cache_dir = WORK_DIR / cache_dir
    return cache_dir / f"fold_{fold}"


def _plain_datasets(
    train_files: list[dict],
    val_files: list[dict],
    train_tf: Compose,
    val_tf: Compose,
) -> tuple[Dataset, Dataset]:
    return Dataset(data=train_files, transform=train_tf), Dataset(data=val_files, transform=val_tf)


def build_loaders(cfg: Cfg, fold: int) -> tuple[DataLoader, DataLoader]:
    splits_dir = Path(str(select(cfg, "paths.splits", str(SPLITS_DIR))))
    train_files, val_files = _load_fold(splits_dir, fold)
    train_tf = build_train_transforms(cfg)
    val_tf = build_val_transforms(cfg)

    cache_rate = float(select(cfg, "data.cache.rate", 0.0) or 0.0)
    cache_workers = int(select(cfg, "data.cache.num_workers", 0) or 0)
    cache_mode = _cache_mode(cfg, cache_rate)

    if cache_mode == "disk" and IS_KAGGLE and not env_bool("ALLOW_DISK_CACHE", False):
        LOGGER.warning(
            "Cache MONAI en disco desactivada en Kaggle; usa ALLOW_DISK_CACHE=1 para forzarla."
        )
        cache_mode = "none"

    if cache_mode == "none":
        train_ds, val_ds = _plain_datasets(train_files, val_files, train_tf, val_tf)
    elif cache_mode == "disk":
        cache_dir = _resolve_cache_dir(cfg, fold)
        if RESET_MONAI_CACHE:
            shutil.rmtree(cache_dir, ignore_errors=True)
        try:
            train_ds = PersistentDataset(
                data=train_files,
                transform=train_tf,
                cache_dir=cache_dir / "train",
            )
            val_ds = PersistentDataset(
                data=val_files,
                transform=val_tf,
                cache_dir=cache_dir / "val",
            )
            LOGGER.info("Usando PersistentDataset en %s", cache_dir)
        except OSError as exc:
            LOGGER.warning(
                "No se pudo usar PersistentDataset en %s (%s); usando Dataset simple.",
                cache_dir,
                exc,
            )
            train_ds, val_ds = _plain_datasets(train_files, val_files, train_tf, val_tf)
    elif cache_rate > 0.0:
        try:
            train_ds = CacheDataset(
                data=train_files,
                transform=train_tf,
                cache_rate=cache_rate,
                num_workers=cache_workers,
                copy_cache=False,
            )
            val_ds = CacheDataset(
                data=val_files,
                transform=val_tf,
                cache_rate=cache_rate,
                num_workers=cache_workers,
                copy_cache=False,
            )
            LOGGER.info("Usando CacheDataset en RAM con cache_rate=%.3f", cache_rate)
        except (OSError, PermissionError) as exc:
            LOGGER.warning(
                "No se pudo usar CacheDataset (%s); usando Dataset simple.",
                exc,
            )
            train_ds, val_ds = _plain_datasets(train_files, val_files, train_tf, val_tf)
    else:
        LOGGER.warning("data.cache.mode=%s con cache_rate <= 0; usando Dataset simple.", cache_mode)
        train_ds, val_ds = _plain_datasets(train_files, val_files, train_tf, val_tf)

    num_workers = int(cfg.training.num_workers)
    pin_memory = bool(cfg.training.pin_memory) and CUDA_AVAILABLE
    train_loader = DataLoader(
        train_ds,
        batch_size=int(cfg.training.batch_size),
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
        worker_init_fn=seed_worker,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=1,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        worker_init_fn=seed_worker,
    )
    return train_loader, val_loader


def predict_volume(model: torch.nn.Module, image: torch.Tensor, cfg: Cfg) -> torch.Tensor:
    def predictor(window: torch.Tensor) -> torch.Tensor:
        output = model(window)
        if isinstance(output, (tuple, list)):
            output = output[0]
        elif isinstance(output, dict):
            output = output.get("logits", next(iter(output.values())))
        return output

    inference_cfg = cfg.training.get("inference", {})
    return sliding_window_inference(
        inputs=image,
        roi_size=tuple(int(value) for value in cfg.training.patch_size),
        sw_batch_size=int(inference_cfg.get("sw_batch_size", 1)),
        predictor=predictor,
        overlap=float(inference_cfg.get("overlap", 0.25)),
        mode=str(inference_cfg.get("mode", "gaussian")),
        padding_mode=str(inference_cfg.get("padding_mode", "constant")),
    )

## Modelo, perdida y metricas

In [6]:
class ViT25D(nn.Module):
    """Segmentador 2.5D: procesa slices 2D con un encoder ViT y recompone el volumen."""

    def __init__(
        self,
        in_channels: int = 1,
        out_channels: int = 2,
        encoder_name: str = "vit_base_patch16_224",
        pretrained: bool = True,
    ) -> None:
        super().__init__()
        self.encoder = timm.create_model(
            encoder_name,
            pretrained=pretrained,
            in_chans=in_channels,
            features_only=True,
            out_indices=(-1,),
            dynamic_img_size=True,
        )
        feature_info = self.encoder.feature_info[-1]
        embed_dim = int(feature_info["num_chs"])
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(embed_dim, 256, kernel_size=4, stride=4),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(256, 64, kernel_size=2, stride=2),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 16, kernel_size=2, stride=2),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, out_channels, kernel_size=1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, channels, height, width, depth = x.shape
        x_2d = x.permute(0, 4, 1, 2, 3).reshape(batch * depth, channels, height, width)
        features = self.encoder(x_2d)
        if isinstance(features, (list, tuple)):
            features = features[-1]
        out_2d = self.decoder(features)
        if out_2d.shape[2:] != (height, width):
            out_2d = functional.interpolate(
                out_2d,
                size=(height, width),
                mode="bilinear",
                align_corners=False,
            )
        return out_2d.view(batch, depth, -1, height, width).permute(0, 2, 3, 4, 1)


def build_vit_25d(cfg: Cfg) -> ViT25D:
    model_cfg = cfg.model if "model" in cfg else cfg
    try:
        return ViT25D(
            in_channels=int(model_cfg.get("in_channels", 1)),
            out_channels=int(model_cfg.get("out_channels", 2)),
            encoder_name=str(model_cfg.get("encoder_name", "vit_base_patch16_224")),
            pretrained=bool(model_cfg.get("pretrained", True)),
        )
    except Exception as exc:
        if bool(model_cfg.get("pretrained", True)) and ALLOW_RANDOM_INIT_FALLBACK:
            LOGGER.warning(
                "No se pudo cargar el encoder preentrenado (%s). Reintentando sin pesos.",
                exc,
            )
            return ViT25D(
                in_channels=int(model_cfg.get("in_channels", 1)),
                out_channels=int(model_cfg.get("out_channels", 2)),
                encoder_name=str(model_cfg.get("encoder_name", "vit_base_patch16_224")),
                pretrained=False,
            )
        raise


MODEL_BUILDERS = {"vit_25d": build_vit_25d}


def build_model(cfg: Cfg) -> torch.nn.Module:
    model_cfg = cfg.model if "model" in cfg else cfg
    name = str(model_cfg.get("name", "vit_25d")).lower()
    try:
        return MODEL_BUILDERS[name](cfg)
    except KeyError as exc:
        raise ValueError(
            f"model.name desconocido={name!r}; esperado {list(MODEL_BUILDERS)}"
        ) from exc


def build_loss(cfg: Cfg | None = None) -> nn.Module:
    loss_cfg = select(cfg or {}, "model.loss", {})
    return DiceFocalLoss(
        include_background=bool(loss_cfg.get("include_background", False)),
        softmax=bool(loss_cfg.get("softmax", True)),
        to_onehot_y=bool(loss_cfg.get("to_onehot_y", True)),
        squared_pred=bool(loss_cfg.get("squared_pred", True)),
        gamma=float(loss_cfg.get("gamma", 2.0)),
        lambda_dice=float(loss_cfg.get("lambda_dice", 1.0)),
        lambda_focal=float(loss_cfg.get("lambda_focal", 1.0)),
    )


def _to_binary_array(value: torch.Tensor | np.ndarray) -> np.ndarray:
    array = value.detach().cpu().numpy() if isinstance(value, torch.Tensor) else np.asarray(value)
    if array.ndim >= 5 and array.shape[1] > 1:
        array = np.argmax(array, axis=1, keepdims=True)
    elif array.ndim >= 5 and array.shape[1] == 1:
        array = array > 0.5
    elif array.ndim == 4:
        array = array[:, None] > 0.5
    return np.asarray(array).astype(bool)


def _surface(mask: np.ndarray) -> np.ndarray:
    if not mask.any():
        return mask.astype(bool)
    eroded = ndimage.binary_erosion(mask)
    return np.logical_xor(mask, eroded)


def _hd95_one(prediction: np.ndarray, label: np.ndarray, spacing: Sequence[float] | None) -> float:
    if not prediction.any() and not label.any():
        return 0.0
    if not prediction.any() or not label.any():
        return float("nan")
    pred_surface = _surface(prediction)
    label_surface = _surface(label)
    sampling = None if spacing is None else tuple(float(value) for value in spacing)
    pred_to_label = ndimage.distance_transform_edt(~label_surface, sampling=sampling)[pred_surface]
    label_to_pred = ndimage.distance_transform_edt(~pred_surface, sampling=sampling)[label_surface]
    distances = np.concatenate([pred_to_label, label_to_pred])
    return float(np.percentile(distances, 95))


def compute_segmentation_metrics(
    prediction: torch.Tensor | np.ndarray,
    label: torch.Tensor | np.ndarray,
    spacing: Sequence[float] | None = None,
) -> dict[str, float]:
    pred_bin = _to_binary_array(prediction)
    label_bin = _to_binary_array(label)
    if pred_bin.shape != label_bin.shape:
        raise ValueError(f"prediction/label shape mismatch: {pred_bin.shape} != {label_bin.shape}")

    dice_scores: list[float] = []
    hd95_scores: list[float] = []
    for pred_one, label_one in zip(pred_bin[:, 0], label_bin[:, 0], strict=True):
        intersection = float(np.logical_and(pred_one, label_one).sum())
        denom = float(pred_one.sum() + label_one.sum())
        dice_scores.append(1.0 if denom == 0.0 else 2.0 * intersection / denom)
        hd95_scores.append(_hd95_one(pred_one, label_one, spacing=spacing))

    hd95_arr = np.asarray(hd95_scores, dtype=np.float64)
    hd95 = float(np.nanmean(hd95_arr)) if not np.isnan(hd95_arr).all() else float("nan")
    return {"dice": float(np.mean(dice_scores)), "hd95": hd95}

## Entrenador actual

In [7]:
def _outputs_dir(cfg: Cfg) -> Path:
    path = Path(str(select(cfg, "paths.outputs", OUTPUTS_ROOT / "manual")))
    path.mkdir(parents=True, exist_ok=True)
    return path


def _save_run_config(out_dir: Path, cfg: Cfg) -> None:
    hydra_dir = out_dir / ".hydra"
    hydra_dir.mkdir(parents=True, exist_ok=True)
    (hydra_dir / "config.json").write_text(
        json.dumps(to_plain(cfg), indent=2) + "\n",
        encoding="utf-8",
    )


def _autocast_context(device: torch.device, enabled: bool):
    if device.type == "cuda" and enabled:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


def _model_output_for_loss(output: torch.Tensor | tuple | list) -> torch.Tensor:
    if isinstance(output, (tuple, list)):
        return output[0]
    return output


def _batch_to_device(
    batch: dict[str, Any], device: torch.device
) -> tuple[torch.Tensor, torch.Tensor]:
    return (
        batch["image"].to(device, non_blocking=True),
        batch["label"].to(device, non_blocking=True),
    )


def _make_optimizer(cfg: Cfg, model: torch.nn.Module) -> torch.optim.Optimizer:
    opt_cfg = cfg.training.optimizer
    name = str(opt_cfg.get("name", "adamw")).lower()
    if name != "adamw":
        raise ValueError(f"optimizer.name desconocido={name!r}; solo se soporta 'adamw'")
    return torch.optim.AdamW(
        model.parameters(),
        lr=float(opt_cfg.get("lr", 1.0e-4)),
        weight_decay=float(opt_cfg.get("weight_decay", 1.0e-5)),
    )


def build_poly_scheduler(
    optimizer: torch.optim.Optimizer, max_steps: int, exp: float = 0.9
) -> LambdaLR:
    if max_steps <= 0:
        raise ValueError("max_steps debe ser > 0")

    def lr_lambda(step: int) -> float:
        clamped = min(max(int(step), 0), int(max_steps))
        return (1.0 - clamped / float(max_steps)) ** float(exp)

    return LambdaLR(optimizer, lr_lambda=lr_lambda)



def build_cosine_warmup_scheduler(
    optimizer: torch.optim.Optimizer, max_steps: int, warmup_steps: int = 500
) -> LambdaLR:
    if max_steps <= 0:
        raise ValueError("max_steps debe ser > 0")

    def lr_lambda(step: int) -> float:
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))

        progress = float(step - warmup_steps) / float(max(1, max_steps - warmup_steps))
        progress = min(max(progress, 0.0), 1.0)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    return LambdaLR(optimizer, lr_lambda=lr_lambda)


def _finite_or_none(value: float) -> float | None:
    return float(value) if math.isfinite(float(value)) else None


def _loader_len(loader: Iterable) -> int:
    try:
        return max(len(loader), 1)  # type: ignore[arg-type]
    except TypeError as exc:
        raise ValueError("El entrenamiento por epocas requiere un train_loader con len().") from exc


def _resolve_resume_path(raw_path: str, out_dir: Path) -> Path:
    if raw_path.lower() in {"last", "best"}:
        return out_dir / "checkpoints" / f"{raw_path.lower()}.pt"
    path = Path(raw_path).expanduser()
    if not path.is_absolute():
        path = Path.cwd() / path
    return path


def _optimizer_state_to_device(optimizer: torch.optim.Optimizer, device: torch.device) -> None:
    for state in optimizer.state.values():
        for key, value in list(state.items()):
            if isinstance(value, torch.Tensor):
                state[key] = value.to(device)


def _write_metrics_csv(path: Path, rows: list[dict[str, float | int]]) -> None:
    if not rows:
        return
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def _read_metrics_csv(path: Path, max_step: int | None = None) -> list[dict[str, float | int]]:
    if not path.exists():
        return []

    rows: list[dict[str, float | int]] = []
    with path.open(newline="", encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            step = int(float(row["step"]))
            if max_step is not None and step > max_step:
                continue
            rows.append(
                {
                    "epoch": int(float(row.get("epoch", 0))),
                    "step": step,
                    "train_loss": float(row["train_loss"]),
                    "lr": float(row["lr"]),
                    "val_dice": float(row["val_dice"]),
                    "val_hd95": float(row["val_hd95"]),
                }
            )
    return rows


def _best_from_rows(rows: list[dict[str, float | int]]) -> tuple[int, int, float, float]:
    best_epoch = 0
    best_step = 0
    best_dice = -1.0
    best_hd95 = float("nan")
    for row in rows:
        dice = float(row["val_dice"])
        if dice > best_dice:
            best_epoch = int(row["epoch"])
            best_step = int(row["step"])
            best_dice = dice
            best_hd95 = float(row["val_hd95"])
    return best_epoch, best_step, best_dice, best_hd95


@torch.no_grad()
def _validate(
    cfg: Cfg,
    model: torch.nn.Module,
    val_loader: Iterable,
    device: torch.device,
) -> dict[str, float]:
    model.eval()
    spacing = select(cfg, "data.target_spacing", None)
    dice_values: list[float] = []
    hd95_values: list[float] = []

    for batch in val_loader:
        image, label = _batch_to_device(batch, device)
        logits = predict_volume(model, image, cfg)
        metrics = compute_segmentation_metrics(logits, label, spacing=spacing)
        dice_values.append(metrics["dice"])
        if math.isfinite(metrics["hd95"]):
            hd95_values.append(metrics["hd95"])

    model.train()
    return {
        "val_dice": float(sum(dice_values) / max(len(dice_values), 1)),
        "val_hd95": float(sum(hd95_values) / len(hd95_values)) if hd95_values else float("nan"),
    }


def _save_checkpoint(
    path: Path,
    cfg: Cfg,
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.LRScheduler,
    scaler: GradScaler,
    *,
    epoch: int,
    step: int,
    metrics: dict[str, float],
    best_epoch: int,
    best_step: int,
    best_metrics: dict[str, float],
    epochs_without_improvement: int,
    unfreeze_done: bool,
) -> None:
    torch.save(
        {
            "epoch": int(epoch),
            "step": int(step),
            "metrics": metrics,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_epoch": int(best_epoch),
            "best_step": int(best_step),
            "best_metrics": best_metrics,
            "epochs_without_improvement": int(epochs_without_improvement),
            "unfreeze_done": bool(unfreeze_done),
            "cfg": to_plain(cfg),
        },
        path,
    )


def _torch_load_checkpoint(path: Path) -> dict:
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


class Trainer:
    _ENCODER_NAME_MARKERS = (
        "encoder",
        "patch_embed",
        "swinvit",
        "swin_vit",
        "layers1",
        "layers2",
        "layers3",
        "layers4",
        "convinit",
        "down_layers",
        "downsample",
        "bottleneck",
    )

    def __init__(
        self,
        cfg: Cfg,
        model: torch.nn.Module,
        loaders: tuple[Iterable, Iterable],
        *,
        unfreeze_epoch: int = -1,
        unfreeze_lr_factor: float = 1.0,
    ) -> None:
        set_global_determinism(int(select(cfg, "seed", 42)))

        self.cfg = cfg
        self.train_loader, self.val_loader = loaders
        self.device = torch.device("cuda" if CUDA_AVAILABLE else "cpu")
        self.model = model.to(self.device)
        self.model.train()

        self.grad_accum_steps = max(int(select(cfg, "experiment.grad_accum_steps", 1)), 1)
        self.steps_per_epoch = max(
            math.ceil(_loader_len(self.train_loader) / self.grad_accum_steps),
            1,
        )
        self.max_epochs = self._resolve_max_epochs()
        self.max_steps = self.max_epochs * self.steps_per_epoch
        self.val_every_epochs = max(
            int(
                select(
                    cfg, "training.val_every_epochs", select(cfg, "experiment.val_every_epochs", 1)
                )
            ),
            1,
        )
        self.val_every_steps = int(select(cfg, "experiment.val_every", 0))
        self.log_every_steps = max(int(select(cfg, "experiment.log_every", 50)), 1)
        self.patience_epochs = max(int(select(cfg, "experiment.patience", 20)), 1)
        self.fixed_epochs = bool(
            select(
                cfg, "experiment.fixed_epochs", select(cfg, "experiment.fixed_iterations", False)
            )
        )

        self.unfreeze_epoch = int(unfreeze_epoch)
        self.unfreeze_lr_factor = float(unfreeze_lr_factor)
        if self.unfreeze_lr_factor <= 0.0:
            raise ValueError("unfreeze_lr_factor debe ser > 0")
        self._unfreeze_done = self.unfreeze_epoch < 0
        if self.unfreeze_epoch > 0:
            self._freeze_encoder()

        self.optimizer = _make_optimizer(cfg, self.model)
        scheduler_name = str(cfg.training.scheduler.get("name", "poly")).lower()
        if scheduler_name == "cosine_warmup":
            self.scheduler = build_cosine_warmup_scheduler(
                self.optimizer,
                max_steps=self.max_steps,
                warmup_steps=int(cfg.training.scheduler.get("warmup_steps", 500)),
            )
        else:
            self.scheduler = build_poly_scheduler(
                self.optimizer,
                max_steps=self.max_steps,
                exp=float(cfg.training.scheduler.get("exp", 0.9)),
            )
        self.base_loss = build_loss(cfg)
        self.amp_enabled = bool(cfg.training.get("amp", False)) and self.device.type == "cuda"
        self.scaler = GradScaler("cuda", enabled=self.amp_enabled)

        self.out_dir = _outputs_dir(cfg)
        self.ckpt_dir = self.out_dir / "checkpoints"
        self.ckpt_dir.mkdir(parents=True, exist_ok=True)
        self.metrics_path = self.out_dir / "metrics.csv"
        self.summary_path = self.out_dir / "summary.json"
        self.best_ckpt = self.ckpt_dir / "best.pt"
        self.last_ckpt = self.ckpt_dir / "last.pt"
        _save_run_config(self.out_dir, cfg)
        self.wandb_run = self._init_wandb()

    def _resolve_max_epochs(self) -> int:
        configured = select(
            self.cfg, "training.max_epochs", select(self.cfg, "experiment.max_epochs", None)
        )
        if configured is not None:
            return max(int(configured), 1)

        max_iterations = select(
            self.cfg,
            "training.sanity.max_iterations",
            select(self.cfg, "experiment.max_iterations", None),
        )
        if max_iterations is not None:
            return max(math.ceil(int(max_iterations) / self.steps_per_epoch), 1)
        return 100

    def _init_wandb(self):
        if not wandb_enabled():
            return None
        try:
            import wandb

            return wandb.init(project="lungseg", config=to_plain(self.cfg), dir=str(self.out_dir))
        except Exception as exc:
            LOGGER.warning("W&B solicitado, pero no se pudo inicializar: %s", exc)
            return None

    def _iter_train_batches(self) -> Iterator[dict[str, Any]]:
        if bool(select(self.cfg, "training.sanity.overfit_one_batch", False)):
            yield next(iter(self.train_loader))
            return
        yield from self.train_loader

    def _epoch_batches(self) -> int:
        if bool(select(self.cfg, "training.sanity.overfit_one_batch", False)):
            return 1
        return _loader_len(self.train_loader)

    def _is_encoder_parameter(self, name: str) -> bool:
        lowered = name.lower()
        return any(marker in lowered for marker in self._ENCODER_NAME_MARKERS)

    def _freeze_encoder(self) -> None:
        for name, param in self.model.named_parameters():
            if self._is_encoder_parameter(name):
                param.requires_grad = False

    def _unfreeze_model(self) -> None:
        if self._unfreeze_done:
            return
        for param in self.model.parameters():
            param.requires_grad = True
        for idx, group in enumerate(self.optimizer.param_groups):
            group["lr"] = float(group["lr"]) * self.unfreeze_lr_factor
            if "initial_lr" in group:
                group["initial_lr"] = float(group["initial_lr"]) * self.unfreeze_lr_factor
            if hasattr(self.scheduler, "base_lrs") and idx < len(self.scheduler.base_lrs):
                self.scheduler.base_lrs[idx] = (
                    float(self.scheduler.base_lrs[idx]) * self.unfreeze_lr_factor
                )
        if hasattr(self.scheduler, "_last_lr"):
            self.scheduler._last_lr = [float(group["lr"]) for group in self.optimizer.param_groups]
        self._unfreeze_done = True

    def _optimizer_step(self) -> None:
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad(set_to_none=True)
        self.scheduler.step()

    def _load_resume_state(self) -> tuple[int, int, int, int, float, float, int, dict[str, float]]:
        resume_from = select(self.cfg, "training.resume_from", None)
        if not resume_from:
            return (
                0,
                0,
                0,
                0,
                -1.0,
                float("nan"),
                0,
                {"val_dice": float("nan"), "val_hd95": float("nan")},
            )

        resume_path = _resolve_resume_path(str(resume_from), self.out_dir)
        checkpoint = _torch_load_checkpoint(resume_path)
        self.model.load_state_dict(checkpoint["model_state_dict"])

        if "optimizer_state_dict" in checkpoint:
            self.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            _optimizer_state_to_device(self.optimizer, self.device)
        if "scheduler_state_dict" in checkpoint:
            self.scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        if "scaler_state_dict" in checkpoint:
            self.scaler.load_state_dict(checkpoint["scaler_state_dict"])

        start_epoch = int(checkpoint.get("epoch", -1)) + 1
        global_step = int(checkpoint.get("step", 0))
        self._unfreeze_done = bool(checkpoint.get("unfreeze_done", self._unfreeze_done))
        if (
            not self._unfreeze_done
            and self.unfreeze_epoch >= 0
            and start_epoch > self.unfreeze_epoch
        ):
            self._unfreeze_model()
        elif self._unfreeze_done:
            for param in self.model.parameters():
                param.requires_grad = True

        rows = _read_metrics_csv(self.metrics_path, max_step=global_step)
        best_epoch, best_step, best_dice, best_hd95 = _best_from_rows(rows)
        if "best_metrics" in checkpoint:
            best_metrics = checkpoint["best_metrics"]
            best_epoch = int(checkpoint.get("best_epoch", best_epoch))
            best_step = int(checkpoint.get("best_step", best_step))
            best_dice = float(best_metrics.get("val_dice", best_dice))
            best_hd95 = float(best_metrics.get("val_hd95", best_hd95))
        last_metrics = dict(
            checkpoint.get("metrics", {"val_dice": float("nan"), "val_hd95": float("nan")})
        )
        stale_epochs = int(checkpoint.get("epochs_without_improvement", 0))
        LOGGER.info("Reanudando entrenamiento desde %s en epoch=%d", resume_path, start_epoch)
        return (
            start_epoch,
            global_step,
            best_epoch,
            best_step,
            best_dice,
            best_hd95,
            stale_epochs,
            last_metrics,
        )

    def fit(self) -> dict[str, Any]:
        (
            start_epoch,
            global_step,
            best_epoch,
            best_step,
            best_dice,
            best_hd95,
            stale_epochs,
            last_metrics,
        ) = self._load_resume_state()
        rows = _read_metrics_csv(self.metrics_path, max_step=global_step)

        for current_epoch in range(start_epoch, self.max_epochs):
            if current_epoch == self.unfreeze_epoch:
                self._unfreeze_model()

            self.model.train()
            self.optimizer.zero_grad(set_to_none=True)
            running_loss = 0.0
            n_loss_terms = 0
            pending_backward = 0
            epoch_batches = self._epoch_batches()

            for batch_idx, batch in enumerate(self._iter_train_batches(), start=1):
                image, label = _batch_to_device(batch, self.device)
                with _autocast_context(self.device, self.amp_enabled):
                    output = _model_output_for_loss(self.model(image))
                    loss = self.base_loss(output, label) / self.grad_accum_steps

                self.scaler.scale(loss).backward()
                running_loss += float(loss.detach().cpu()) * self.grad_accum_steps
                n_loss_terms += 1
                pending_backward += 1

                if pending_backward < self.grad_accum_steps and batch_idx < epoch_batches:
                    continue

                self._optimizer_step()
                global_step += 1
                pending_backward = 0

                if global_step % self.log_every_steps == 0 or global_step == 1:
                    lr = float(self.optimizer.param_groups[0]["lr"])
                    mean_loss = running_loss / max(n_loss_terms, 1)
                    LOGGER.info(
                        "epoch=%d step=%d loss=%.4f lr=%.6g",
                        current_epoch,
                        global_step,
                        mean_loss,
                        lr,
                    )
                    if self.wandb_run is not None:
                        self.wandb_run.log(
                            {
                                "train/loss": mean_loss,
                                "lr": lr,
                                "epoch": current_epoch,
                                "step": global_step,
                            }
                        )
                
                # VALIDACIÓN POR PASOS (Super-convergencia)
                if self.val_every_steps > 0 and global_step % self.val_every_steps == 0:
                    self.model.eval()
                    mean_train_loss = running_loss / max(n_loss_terms, 1)
                    last_metrics = _validate(self.cfg, self.model, self.val_loader, self.device)
                    row = {
                        "epoch": current_epoch,
                        "step": global_step,
                        "train_loss": mean_train_loss,
                        "lr": float(self.optimizer.param_groups[0]["lr"]),
                        "val_dice": last_metrics["val_dice"],
                        "val_hd95": last_metrics["val_hd95"],
                    }
                    rows.append(row)
                    _write_metrics_csv(self.metrics_path, rows)

                    if self.wandb_run is not None:
                        self.wandb_run.log(
                            {
                                "val/dice": last_metrics["val_dice"],
                                "val/hd95": last_metrics["val_hd95"],
                                "epoch": current_epoch,
                                "step": global_step,
                            }
                        )

                    if last_metrics["val_dice"] > best_dice:
                        best_epoch, best_step, best_dice, best_hd95 = (
                            current_epoch,
                            global_step,
                            last_metrics["val_dice"],
                            last_metrics["val_hd95"],
                        )
                        stale_epochs = 0
                        _save_checkpoint(
                            self.best_ckpt,
                            self.cfg,
                            self.model,
                            self.optimizer,
                            self.scheduler,
                            self.scaler,
                            epoch=current_epoch,
                            step=global_step,
                            metrics=last_metrics,
                            best_epoch=best_epoch,
                            best_step=best_step,
                            best_metrics={"val_dice": best_dice, "val_hd95": best_hd95},
                            epochs_without_improvement=stale_epochs,
                            unfreeze_done=self._unfreeze_done,
                        )
                    else:
                        stale_epochs += 1
                    
                    self.model.train()

                    if not self.fixed_epochs and stale_epochs >= self.patience_epochs:
                        LOGGER.info("Early stopping agresivo en paso %d", global_step)
                        return {
                            "best_epoch": int(best_epoch),
                            "best_val_dice": float(best_dice),
                            "checkpoint_path": str(self.best_ckpt),
                        }

            should_validate = (
                current_epoch + 1
            ) % self.val_every_epochs == 0 or current_epoch == self.max_epochs - 1
            
            # Evitar doble validación si ya ocurrió por pasos en este mismo step
            if should_validate and (self.val_every_steps <= 0 or global_step % self.val_every_steps != 0):
                self.model.eval()
                mean_train_loss = running_loss / max(n_loss_terms, 1)
                last_metrics = _validate(self.cfg, self.model, self.val_loader, self.device)
                last_metrics = _validate(self.cfg, self.model, self.val_loader, self.device)
                row = {
                    "epoch": current_epoch,
                    "step": global_step,
                    "train_loss": mean_train_loss,
                    "lr": float(self.optimizer.param_groups[0]["lr"]),
                    "val_dice": last_metrics["val_dice"],
                    "val_hd95": last_metrics["val_hd95"],
                }
                rows.append(row)
                _write_metrics_csv(self.metrics_path, rows)

                if self.wandb_run is not None:
                    self.wandb_run.log(
                        {
                            "val/dice": last_metrics["val_dice"],
                            "val/hd95": last_metrics["val_hd95"],
                            "epoch": current_epoch,
                            "step": global_step,
                        }
                    )

                if last_metrics["val_dice"] > best_dice:
                    best_epoch = current_epoch
                    best_step = global_step
                    best_dice = last_metrics["val_dice"]
                    best_hd95 = last_metrics["val_hd95"]
                    stale_epochs = 0
                    _save_checkpoint(
                        self.best_ckpt,
                        self.cfg,
                        self.model,
                        self.optimizer,
                        self.scheduler,
                        self.scaler,
                        epoch=current_epoch,
                        step=global_step,
                        metrics=last_metrics,
                        best_epoch=best_epoch,
                        best_step=best_step,
                        best_metrics={"val_dice": best_dice, "val_hd95": best_hd95},
                        epochs_without_improvement=stale_epochs,
                        unfreeze_done=self._unfreeze_done,
                    )
                else:
                    stale_epochs += self.val_every_epochs

            _save_checkpoint(
                self.last_ckpt,
                self.cfg,
                self.model,
                self.optimizer,
                self.scheduler,
                self.scaler,
                epoch=current_epoch,
                step=global_step,
                metrics=last_metrics,
                best_epoch=best_epoch,
                best_step=best_step,
                best_metrics={"val_dice": best_dice, "val_hd95": best_hd95},
                epochs_without_improvement=stale_epochs,
                unfreeze_done=self._unfreeze_done,
            )

            if not self.fixed_epochs and stale_epochs >= self.patience_epochs:
                LOGGER.info("Early stopping en epoch=%d", current_epoch)
                break

        summary = {
            "best_epoch": int(best_epoch),
            "best_step": int(best_step),
            "best_val_dice": _finite_or_none(best_dice),
            "best_val_hd95": _finite_or_none(best_hd95),
            "checkpoint_path": str(self.best_ckpt),
            "last_step": int(global_step),
            "last_checkpoint_path": str(self.last_ckpt),
            "metrics_path": str(self.metrics_path),
        }
        self.summary_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
        if self.wandb_run is not None:
            self.wandb_run.finish()
        return summary

    def run(self) -> dict[str, Any]:
        return self.fit()


def train_iters(
    cfg: Cfg,
    model: torch.nn.Module,
    loaders: tuple[Iterable, Iterable],
) -> dict[str, Any]:
    return Trainer(
        cfg,
        model,
        loaders,
        unfreeze_epoch=int(select(cfg, "training.unfreeze_epoch", -1)),
        unfreeze_lr_factor=float(select(cfg, "training.unfreeze_lr_factor", 1.0)),
    ).fit()


def train_segmentation_run(cfg: Cfg) -> dict[str, Any]:
    model = build_model(cfg)
    loaders = build_loaders(cfg, fold=int(cfg.fold))
    return train_iters(cfg, model, loaders)


## Phase 6: ablacion y analisis

In [8]:
def _sample_train_records(records: list[dict], fraction: float, seed: int) -> list[dict]:
    if fraction >= 1.0:
        return list(records)
    if fraction <= 0.0:
        raise ValueError("data_fraction debe ser > 0")
    rng = np.random.default_rng(seed)
    selected: list[dict] = []
    strata = sorted({int(record.get("stratum", 0)) for record in records})
    for stratum in strata:
        bucket = [record for record in records if int(record.get("stratum", 0)) == stratum]
        n_keep = max(1, round(len(bucket) * fraction))
        indices = np.sort(rng.choice(len(bucket), size=min(n_keep, len(bucket)), replace=False))
        selected.extend(bucket[int(index)] for index in indices)
    return sorted(selected, key=lambda record: record["patient_id"])


def _write_fractional_split(cfg: Cfg, fraction: float, seed: int) -> Path:
    fold = int(select(cfg, "fold", 0))
    source_dir = Path(str(select(cfg, "paths.splits", str(SPLITS_DIR))))
    source_path = source_dir / f"fold_{fold}.json"
    payload = json.loads(source_path.read_text(encoding="utf-8"))
    payload["train"] = _sample_train_records(payload["train"], fraction=fraction, seed=seed)
    payload["n_train"] = len(payload["train"])
    payload["ablation"] = {"data_fraction": fraction, "seed": seed}

    out_dir = _outputs_dir(cfg) / "ablation_splits"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"fold_{fold}_frac_{fraction:g}_seed_{seed}.json"
    out_path.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")

    active_dir = out_dir / f"active_frac_{fraction:g}_seed_{seed}"
    active_dir.mkdir(parents=True, exist_ok=True)
    (active_dir / f"fold_{fold}.json").write_text(
        json.dumps(payload, indent=2) + "\n",
        encoding="utf-8",
    )
    return active_dir


def run_ablation_cell(cfg: Cfg) -> dict[str, Any]:
    seed = int(select(cfg, "seed", select(cfg, "experiment.seed", 42)))
    fraction = float(select(cfg, "data_fraction", select(cfg, "experiment.data_fraction", 1.0)))
    augment_regime = str(
        select(cfg, "aug_regime", select(cfg, "training.augment_regime", "standard"))
    )

    cell_cfg = cfgify(copy.deepcopy(to_plain(cfg)))
    update_cfg(cell_cfg, "seed", seed)
    update_cfg(cell_cfg, "training.augment_regime", augment_regime)
    update_cfg(cell_cfg, "experiment.fixed_iterations", True)
    split_dir = _write_fractional_split(cell_cfg, fraction=fraction, seed=seed)
    update_cfg(cell_cfg, "paths.splits", str(split_dir))

    summary = train_segmentation_run(cell_cfg)
    result = {
        "seed": seed,
        "data_fraction": fraction,
        "augment_regime": augment_regime,
        **summary,
    }
    out_dir = _outputs_dir(cell_cfg) / "ablation"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"frac_{fraction:g}_aug_{augment_regime}_seed_{seed}.json"
    out_path.write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
    result["result_path"] = str(out_path)
    return result


def _markdown_table(df: pd.DataFrame) -> list[str]:
    columns = list(df.columns)
    lines = ["| " + " | ".join(columns) + " |", "| " + " | ".join(["---"] * len(columns)) + " |"]
    for _, row in df.iterrows():
        values = []
        for col in columns:
            value = row[col]
            values.append(f"{value:.4g}" if isinstance(value, float) else str(value))
        lines.append("| " + " | ".join(values) + " |")
    return lines


def analyze_ablation(outputs_dir: Path) -> Path:
    outputs_dir = Path(outputs_dir)
    ablation_dir = outputs_dir / "ablation"
    rows = [
        json.loads(path.read_text(encoding="utf-8")) for path in sorted(ablation_dir.glob("*.json"))
    ]
    if not rows:
        raise FileNotFoundError(f"No hay JSON de ablacion en {ablation_dir}")

    df = pd.DataFrame(rows)
    if "best_val_hd95" not in df.columns:
        df["best_val_hd95"] = np.nan
    summary = (
        df.groupby(["data_fraction", "augment_regime"], as_index=False)
        .agg(
            dice_median=("best_val_dice", "median"),
            dice_q1=("best_val_dice", lambda series: series.quantile(0.25)),
            dice_q3=("best_val_dice", lambda series: series.quantile(0.75)),
            hd95_median=("best_val_hd95", "median"),
            hd95_q1=("best_val_hd95", lambda series: series.quantile(0.25)),
            hd95_q3=("best_val_hd95", lambda series: series.quantile(0.75)),
            n=("best_val_dice", "count"),
        )
        .sort_values(["data_fraction", "augment_regime"])
    )
    summary_path = outputs_dir / "ablation_summary.csv"
    summary.to_csv(summary_path, index=False)

    plot_path = outputs_dir / "ablation_violin.png"
    try:
        import matplotlib.pyplot as plt

        plot_df = df.copy()
        plot_df["cell"] = (
            plot_df["data_fraction"].astype(str) + " / " + plot_df["augment_regime"].astype(str)
        )
        fig, ax = plt.subplots(figsize=(10, 4))
        ordered = sorted(plot_df["cell"].unique())
        ax.violinplot(
            [plot_df.loc[plot_df["cell"] == cell, "best_val_dice"].dropna() for cell in ordered]
        )
        ax.set_xticks(range(1, len(ordered) + 1), ordered, rotation=30, ha="right")
        ax.set_ylabel("Best val Dice")
        ax.set_title("Phase 6 ablation")
        fig.tight_layout()
        fig.savefig(plot_path, dpi=160)
        plt.close(fig)
    except Exception as exc:
        LOGGER.warning("No se pudo generar violin plot: %s", exc)
        plot_path = Path("")

    wilcoxon_rows = []
    for fraction in sorted(df["data_fraction"].unique()):
        sub = df[df["data_fraction"] == fraction]
        pivot = sub.pivot_table(index="seed", columns="augment_regime", values="best_val_dice")
        if {"none", "standard"}.issubset(pivot.columns) and len(pivot.dropna()) >= 2:
            try:
                stat, p_value = wilcoxon(pivot["none"], pivot["standard"])
                wilcoxon_rows.append((fraction, float(stat), float(p_value)))
            except ValueError:
                wilcoxon_rows.append((fraction, float("nan"), float("nan")))

    report_path = outputs_dir / "REPORT_ABLATION.md"
    lines = [
        "# INFORME_ABLACION",
        "",
        "## Resumen",
        "",
        *_markdown_table(summary),
        "",
        "## Pruebas pareadas de Wilcoxon",
        "",
    ]
    if wilcoxon_rows:
        lines.append("| data_fraction | statistic | p_value |")
        lines.append("|---:|---:|---:|")
        for fraction, stat, p_value in wilcoxon_rows:
            lines.append(f"| {fraction:g} | {stat:.4g} | {p_value:.4g} |")
    else:
        lines.append("No hay suficientes semillas emparejadas para ejecutar Wilcoxon.")
    lines.extend(["", f"- Summary CSV: `{summary_path}`"])
    if str(plot_path):
        lines.append(f"- Violin plot: `{plot_path}`")
    report_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return report_path

## Phase 5 opcional: radiomica LIDC

In [9]:
REQUIRED_LIDC_COLUMNS = {"patient_id", "nodule_id", "image", "mask_gt", "malignancy_median"}


def locate_lidc_manifest() -> Path | None:
    candidates = [
        LIDC_MANIFEST,
        WORK_DIR / "data" / "processed" / "lidc" / "nodule_manifest.csv",
        Path.cwd() / "data" / "processed" / "lidc" / "nodule_manifest.csv",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    input_dir = Path("/kaggle/input")
    if input_dir.exists():
        matches = sorted(input_dir.rglob("nodule_manifest.csv"))
        if matches:
            return matches[0].resolve()
    return None


def ensure_phase5_dependencies() -> None:
    if importlib.util.find_spec("radiomics") is None:
        _pip_install(["git+https://github.com/AIM-Harvard/pyradiomics.git"])
    if importlib.util.find_spec("xgboost") is None:
        _pip_install(["xgboost>=2.0"])


def extract_radiomics(image_path: Path, mask_path: Path) -> dict[str, float]:
    try:
        from radiomics import featureextractor
    except ImportError as exc:
        raise ImportError("PyRadiomics es necesario para Phase 5.") from exc

    image_path = Path(image_path)
    mask_path = Path(mask_path)
    if not image_path.exists():
        raise FileNotFoundError(f"Imagen radiomica no encontrada: {image_path}")
    if not mask_path.exists():
        raise FileNotFoundError(f"Mascara radiomica no encontrada: {mask_path}")

    extractor = featureextractor.RadiomicsFeatureExtractor(
        resampledPixelSpacing=[1.0, 1.0, 1.0],
        interpolator="sitkBSpline",
        binWidth=25,
        label=1,
    )
    extractor.enableImageTypes(Original={}, LoG={"sigma": [1.0, 2.0, 3.0]}, Wavelet={})
    result = extractor.execute(str(image_path), str(mask_path))
    features: dict[str, float] = {}
    for key, value in result.items():
        if key.startswith("diagnostics_"):
            continue
        try:
            features[key] = float(value)
        except (TypeError, ValueError):
            continue
    return features


def _read_manifest(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"LIDC manifest no encontrado: {path}")
    if path.suffix.lower() == ".json":
        payload = json.loads(path.read_text(encoding="utf-8"))
        rows = payload["nodules"] if isinstance(payload, dict) and "nodules" in payload else payload
        return pd.DataFrame(rows)
    return pd.read_csv(path)


def _label_from_malignancy(value: float, benign_max: float, malignant_min: float) -> int | None:
    if value >= malignant_min:
        return 1
    if value <= benign_max:
        return 0
    return None


def _resolve_path(value: object, base_dir: Path) -> Path:
    path = Path(str(value))
    return path if path.is_absolute() else base_dir / path


def _mask_for_row(row: pd.Series, cfg: Cfg, e2e: bool, base_dir: Path) -> Path:
    if not e2e:
        return _resolve_path(row["mask_gt"], base_dir)
    pred_dir_value = select(cfg, "data.pred_masks_dir", None)
    if pred_dir_value is None:
        raise ValueError("e2e=True requiere cfg.data.pred_masks_dir con mascaras predichas")
    pred_dir = Path(str(pred_dir_value))
    candidates = [
        pred_dir / f"{row['patient_id']}_{row['nodule_id']}.nii.gz",
        pred_dir / f"{row['nodule_id']}.nii.gz",
        pred_dir / str(row.get("mask_pred", "")),
    ]
    for candidate in candidates:
        if candidate.name and candidate.exists():
            return candidate
    raise FileNotFoundError(
        "Mascara predicha no encontrada para "
        f"patient_id={row['patient_id']} nodule_id={row['nodule_id']} en {pred_dir}"
    )


def build_radiomic_dataset(cfg: Cfg, e2e: bool = False) -> dict[str, Any]:
    data_name = str(select(cfg, "data.name", "")).lower()
    if data_name in {"task06", "task06_lung", "msd_task06"}:
        raise ValueError("MSD Task06_Lung no tiene etiquetas benigno/maligno; usa LIDC-IDRI")
    if data_name not in {"lidc", "lidc-idri", "lidc_idri"}:
        raise ValueError(f"Dataset de clasificacion no soportado: {data_name!r}")

    manifest_path = Path(str(select(cfg, "data.manifest", "")))
    manifest = _read_manifest(manifest_path)
    missing = sorted(REQUIRED_LIDC_COLUMNS - set(manifest.columns))
    if missing:
        raise ValueError(f"El manifiesto LIDC no tiene columnas requeridas: {missing}")

    benign_max = float(select(cfg, "data.malignancy_consensus.benign_max", 2))
    malignant_min = float(select(cfg, "data.malignancy_consensus.malignant_min", 4))
    rows: list[dict[str, object]] = []
    for _, row in tqdm(manifest.iterrows(), total=len(manifest), desc="Extrayendo radiomica LIDC"):
        label = _label_from_malignancy(float(row["malignancy_median"]), benign_max, malignant_min)
        if label is None:
            continue
        image_path = _resolve_path(row["image"], manifest_path.parent)
        mask_path = _mask_for_row(row, cfg, e2e=e2e, base_dir=manifest_path.parent)
        features = extract_radiomics(image_path, mask_path)
        rows.append(
            {
                "patient_id": str(row["patient_id"]),
                "nodule_id": str(row["nodule_id"]),
                "label": int(label),
                "malignancy_median": float(row["malignancy_median"]),
                "image": str(image_path),
                "mask": str(mask_path),
                **features,
            }
        )

    if not rows:
        raise ValueError(
            "No quedan nodulos benignos/malignos tras descartar malignancy_median == 3"
        )
    table = pd.DataFrame(rows)
    feature_columns = [
        column
        for column in table.columns
        if column not in {"patient_id", "nodule_id", "label", "malignancy_median", "image", "mask"}
    ]
    return {
        "table": table,
        "X": table[feature_columns].to_numpy(dtype=np.float32),
        "y": table["label"].to_numpy(dtype=np.int64),
        "groups": table["patient_id"].to_numpy(dtype=str),
        "feature_names": feature_columns,
    }


def _ece(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 10) -> float:
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    total = len(y_true)
    if total == 0:
        return float("nan")
    ece = 0.0
    for lo, hi in pairwise(bins):
        mask = (y_prob >= lo) & (y_prob < hi if hi < 1.0 else y_prob <= hi)
        if not np.any(mask):
            continue
        confidence = float(np.mean(y_prob[mask]))
        accuracy = float(np.mean(y_true[mask]))
        ece += (float(mask.sum()) / total) * abs(confidence - accuracy)
    return float(ece)


def _safe_auc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_prob))


def _classification_metrics(y_true: np.ndarray, y_prob: np.ndarray) -> dict[str, float]:
    y_pred = (y_prob >= 0.5).astype(int)
    return {
        "auc": _safe_auc(y_true, y_prob),
        "balanced_acc": float(balanced_accuracy_score(y_true, y_pred)),
        "brier": float(brier_score_loss(y_true, y_prob)),
        "ece": _ece(y_true, y_prob),
    }


def _n_splits(y: np.ndarray, groups: np.ndarray, requested: int) -> int:
    unique_groups = np.unique(groups)
    if len(unique_groups) < 2:
        raise ValueError("La CV de clasificacion requiere al menos dos grupos de paciente")
    min_class = int(np.min(np.bincount(y.astype(int))))
    n_splits = min(int(requested), len(unique_groups), min_class)
    if n_splits < 2:
        raise ValueError("La CV de clasificacion requiere al menos dos muestras por clase")
    return n_splits


def _models(seed: int) -> dict[str, object]:
    models: dict[str, object] = {
        "rf": RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced",
            random_state=seed,
            n_jobs=-1,
        ),
        "lasso": LogisticRegression(
            l1_ratio=1.0,
            solver="liblinear",
            class_weight="balanced",
            max_iter=1000,
            random_state=seed,
        ),
        "mlp": MLPClassifier(
            hidden_layer_sizes=(32,),
            alpha=1.0e-3,
            max_iter=500,
            random_state=seed,
        ),
    }
    try:
        from xgboost import XGBClassifier

        models["xgb"] = XGBClassifier(
            n_estimators=200,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric="logloss",
            random_state=seed,
        )
    except ImportError:
        pass
    return models


def _make_pipeline(estimator: object, k: int, seed: int) -> Pipeline:
    return Pipeline(
        steps=[
            ("scale", RobustScaler()),
            ("select", SelectKBest(partial(mutual_info_classif, random_state=seed), k=k)),
            ("clf", estimator),
        ]
    )


def _positive_proba(model: Pipeline, features: np.ndarray) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        return model.predict_proba(features)[:, 1]
    decision = model.decision_function(features)
    return 1.0 / (1.0 + np.exp(-decision))


def evaluate_pipeline(features: np.ndarray, y: np.ndarray, groups: np.ndarray, cfg: Cfg) -> dict:
    features = np.asarray(features, dtype=np.float32)
    y = np.asarray(y, dtype=np.int64)
    groups = np.asarray(groups)
    if features.ndim != 2:
        raise ValueError(f"features debe ser 2D, shape actual {features.shape}")
    if len(features) != len(y) or len(features) != len(groups):
        raise ValueError("features, y y groups deben tener la misma longitud")

    seed = int(select(cfg, "seed", 42))
    requested_splits = int(select(cfg, "data.n_folds", 5))
    n_splits = _n_splits(y, groups, requested_splits)
    k = min(20, features.shape[1])
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    results: dict[str, dict] = {}
    for name, estimator in _models(seed).items():
        oof = np.zeros(len(y), dtype=np.float64)
        fold_rows = []
        for fold, (train_idx, val_idx) in enumerate(splitter.split(features, y, groups=groups)):
            pipe = _make_pipeline(clone(estimator), k=k, seed=seed)
            pipe.fit(features[train_idx], y[train_idx])
            proba = _positive_proba(pipe, features[val_idx])
            oof[val_idx] = proba
            fold_rows.append({"fold": fold, **_classification_metrics(y[val_idx], proba)})
        results[name] = {
            **_classification_metrics(y, oof),
            "n_splits": n_splits,
            "k_features": k,
            "folds": fold_rows,
        }
    return results


def evaluate_size_only(volumes: np.ndarray, y: np.ndarray, groups: np.ndarray) -> dict:
    volumes = np.asarray(volumes, dtype=np.float32).reshape(-1, 1)
    y = np.asarray(y, dtype=np.int64)
    groups = np.asarray(groups)
    n_splits = _n_splits(y, groups, requested=5)
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof = np.zeros(len(y), dtype=np.float64)
    folds = []
    for fold, (train_idx, val_idx) in enumerate(splitter.split(volumes, y, groups=groups)):
        pipe = Pipeline(
            steps=[
                ("scale", RobustScaler()),
                (
                    "clf",
                    LogisticRegression(
                        class_weight="balanced",
                        solver="liblinear",
                        random_state=42,
                    ),
                ),
            ]
        )
        pipe.fit(volumes[train_idx], y[train_idx])
        proba = pipe.predict_proba(volumes[val_idx])[:, 1]
        oof[val_idx] = proba
        folds.append({"fold": fold, **_classification_metrics(y[val_idx], proba)})
    return {
        **_classification_metrics(y, oof),
        "n_splits": n_splits,
        "folds": folds,
    }


def run_phase5(e2e: bool = False) -> dict | None:
    manifest = locate_lidc_manifest()
    if manifest is None:
        print(
            f"Saltando Phase 5: no existe {LIDC_MANIFEST} ni se encontro "
            "nodule_manifest.csv en /kaggle/input."
        )
        print("Task06 no tiene etiquetas benigno/maligno; Phase 5 requiere LIDC-IDRI.")
        return None

    ensure_phase5_dependencies()
    cfg = make_cfg(experiment_name="phase4_full", outputs=OUTPUTS_ROOT / "phase5", data_name="lidc")
    cfg.data.manifest = str(manifest)
    dataset = build_radiomic_dataset(cfg, e2e=e2e)
    full = evaluate_pipeline(dataset["X"], dataset["y"], dataset["groups"], cfg)
    volumes = dataset["table"].get("original_shape_VoxelVolume")
    if volumes is None:
        volume_candidates = dataset["table"].filter(like="VoxelVolume")
        if volume_candidates.empty:
            raise ValueError("La linea base de tamano necesita una caracteristica VoxelVolume")
        volumes = volume_candidates.iloc[:, 0]
    size_only = evaluate_size_only(volumes.to_numpy(), dataset["y"], dataset["groups"])
    result = {"full": full, "size_only": size_only, "n_nodules": len(dataset["y"])}
    out_path = Path(str(cfg.paths.outputs)) / "classification_results.json"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
    print(f"Resultados de clasificacion guardados en {out_path}")
    return result

## Pipeline principal

In [10]:
def free_memory() -> None:
    gc.collect()
    if CUDA_AVAILABLE:
        torch.cuda.empty_cache()


def run_phase4_all() -> list[dict[str, Any]]:
    summaries: list[dict[str, Any]] = []
    print(f"==> Phase 4 en folds: {PHASE4_FOLDS}")
    for fold in PHASE4_FOLDS:
        print(f"---- Phase 4 fold {fold} ----")
        free_memory()
        cfg = make_cfg(
            experiment_name="phase4_full",
            fold=fold,
            outputs=OUTPUTS_ROOT / "phase4" / f"fold_{fold}",
        )
        summary = train_segmentation_run(cfg)
        summaries.append({"fold": fold, **summary})
    out_path = OUTPUTS_ROOT / "phase4_summary.json"
    out_path.write_text(json.dumps(summaries, indent=2) + "\n", encoding="utf-8")
    return summaries


def run_phase6_all() -> list[dict[str, Any]]:
    results: list[dict[str, Any]] = []
    print(f"==> Phase 6 ablation en folds: {PHASE6_FOLDS}")
    for fold in PHASE6_FOLDS:
        fold_out = OUTPUTS_ROOT / "phase6" / f"fold_{fold}"
        for fraction in FRACTIONS:
            for augmentation in AUGS:
                for seed in SEEDS:
                    print(
                        "---- Phase 6 "
                        f"fold={fold} fraction={fraction} aug={augmentation} seed={seed} ----"
                    )
                    free_memory()
                    cfg = make_cfg(
                        experiment_name="phase6_ablation",
                        fold=fold,
                        outputs=fold_out,
                        seed=seed,
                    )
                    update_cfg(cfg, "data_fraction", float(fraction))
                    update_cfg(cfg, "aug_regime", str(augmentation))
                    result = run_ablation_cell(cfg)
                    report = analyze_ablation(fold_out)
                    print(f"Ablation cell: {result['result_path']}")
                    print(f"Ablation report: {report}")
                    results.append({"fold": fold, **result})
    out_path = OUTPUTS_ROOT / "phase6_results.json"
    out_path.write_text(json.dumps(results, indent=2) + "\n", encoding="utf-8")
    return results


def print_summaries(root: Path) -> None:
    print(f"==> Summaries en {root}")
    if not root.exists():
        print(f"No existe {root}.")
        return
    found = False
    for path in sorted(root.rglob("summary.json")):
        found = True
        print(f"\n---- {path} ----")
        payload = json.loads(path.read_text(encoding="utf-8"))
        print(json.dumps(payload, indent=2))
    if not found:
        print("No hay summary.json todavia.")

In [ ]:
OUTPUTS_ROOT.mkdir(parents=True, exist_ok=True)
prepare_task06()

print("Pipeline preparado. Ejecutando fases activas...")
free_memory()
phase4_summaries = run_phase4_all() if RUN_PHASE4 else []
phase6_results = run_phase6_all() if RUN_PHASE6 else []
phase5_result = run_phase5(e2e=PHASE5_E2E) if RUN_PHASE5 else None

run_summary = {
    "run_id": RUN_ID,
    "outputs_root": str(OUTPUTS_ROOT),
    "phase4": phase4_summaries,
    "phase6": phase6_results,
    "phase5": phase5_result,
}
(OUTPUTS_ROOT / "pipeline_summary.json").write_text(
    json.dumps(run_summary, indent=2) + "\n",
    encoding="utf-8",
)
print_summaries(OUTPUTS_ROOT)
print("Pipeline completado.")

No se encontro Task06 localmente. Descargando https://msd-for-monai.s3-us-west-2.amazonaws.com/Task06_Lung.tar...
